In [1]:
%cd ..

c:\Users\HP\OneDrive - University of Moratuwa\Desktop\E-Vision-Projects\DB_SQL_GEN


In [2]:
import httpx
from loguru import logger
from src.config import settings

In [3]:
class ChatBotAuthClient:
    def __init__(self):
        self.auth_url = getattr(
            settings,
            "chatbot_auth_url"
        )
        self.api_key = getattr(settings, "chatbot_api_key", "123")

    async def authenticate_user(self, phone_no: str) -> dict:
        headers = {
            "Content-Type": "application/json",
            "CHatBot-Key": self.api_key
        }

        payload = {
            "Phoneno": phone_no
        }

        try:
            async with httpx.AsyncClient(timeout=30) as client:
                response = await client.post(
                    self.auth_url,
                    json=payload,
                    headers=headers
                )

            response.raise_for_status()
            return response.json()

        except Exception as e:
            logger.error(f"AuthenticateUser API failed: {e}")
            raise


chatbot_auth_client = ChatBotAuthClient()

In [4]:
from typing import Any


def get_value(data: dict, *keys, default=None):
    for key in keys:
        if isinstance(data, dict) and key in data:
            return data[key]
    return default


def normalize_list(value: Any) -> list:
    if value is None:
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, str):
        return [x.strip() for x in value.split(",") if x.strip()]

    return []


def extract_allowed_rep_codes(auth_data: dict) -> list[str]:
    """
    Extract allowed RepCodes for SQL filter:
    FIND_IN_SET(shn.Code, @AllowedNodes) > 0

    For Rep response:
      use allowedNodes[].Code

    For Customer response:
      use rep.Code or rep.NodeCode
    """

    if not auth_data:
        return []

    success = get_value(auth_data, "success", "Success", default=True)
    if success is False:
        return []

    user_type = get_value(auth_data, "userType", "UserType", default="")

    allowed_codes = []

    # Rep response: allowedNodes contains accessible hierarchy nodes
    allowed_nodes = get_value(auth_data, "allowedNodes", "AllowedNodes", default=[])

    for node in normalize_list(allowed_nodes):
        if isinstance(node, dict):
            code = get_value(node, "Code", "code", "NodeCode", "nodeCode")
            if code:
                allowed_codes.append(str(code).strip())
        elif isinstance(node, str):
            allowed_codes.append(node.strip())

    # Some APIs may return direct allowed rep code arrays
    for key in ["allowedRepCodes", "AllowedRepCodes", "repCodes", "RepCodes"]:
        for code in normalize_list(auth_data.get(key)):
            if isinstance(code, str):
                allowed_codes.append(code.strip())
            elif isinstance(code, dict):
                value = get_value(code, "Code", "code", "RepCode", "repCode")
                if value:
                    allowed_codes.append(str(value).strip())

    # # Customer response: assigned rep object
    # rep = get_value(auth_data, "rep", "Rep", default=None)
    # if isinstance(rep, dict):
    #     rep_code = get_value(
    #         rep,
    #         "Code",
    #         "code",
    #         "NodeCode",
    #         "nodeCode",
    #         "RepCode",
    #         "repCode"
    #     )
    #     if rep_code:
    #         allowed_codes.append(str(rep_code).strip())

    # Remove duplicates
    return sorted(set(x for x in allowed_codes if x))


def extract_user_role(auth_data: dict) -> str:
    user_type = get_value(auth_data, "userType", "UserType", default=None)

    if user_type:
        return str(user_type).lower()

    if get_value(auth_data, "rep", "Rep"):
        return "rep"

    if get_value(auth_data, "customer", "Customer"):
        return "customer"

    return "unknown"


def extract_user_context(auth_data: dict) -> dict:
    return {
        "user_type": extract_user_role(auth_data),
        "allowed_rep_codes": extract_allowed_rep_codes(auth_data),
        "raw_auth": auth_data
    }

In [5]:
auth_data = await chatbot_auth_client.authenticate_user(phone_no="0718543880")

2026-09-23 07:33:10.767 | ERROR    | __main__:authenticate_user:31 - AuthenticateUser API failed: Client error '404 Not Found' for url 'http://136.243.82.33:8040/ChatBot/AuthenticateUser'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404


HTTPStatusError: Client error '404 Not Found' for url 'http://136.243.82.33:8040/ChatBot/AuthenticateUser'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404

In [8]:
auth_data

NameError: name 'auth_data' is not defined

In [36]:
extract_user_context(auth_data)

{'user_type': 'rep',
 'allowed_rep_codes': ['MATREP001'],
 'raw_auth': {'success': True,
  'userType': 'Rep',
  'rep': {'Id': 3,
   'Code': 'User001',
   'Name': 'Sadeepa Sadaruwan',
   'MobileNo': '0718543880',
   'EMail': None,
   'NodeId': 8,
   'NodeCode': 'MATREP001',
   'NodeName': 'Matara Rep 01',
   'PdaImei': None},
  'allowedNodeIds': [8],
  'allowedTerritoryIds': [8],
  'allowedRepIds': [8],
  'allowedDistributorIds': [3742],
  'allowedNodes': [{'Id': 8,
    'Code': 'MATREP001',
    'Name': 'Matara Rep 01',
    'LevelType': 3,
    'PdaImei': None,
    'UserId': 3,
    'UserName': 'Sadeepa Sadaruwan'}]}}

In [6]:
# src/tools/system_login_client.py

import json
import httpx
from loguru import logger

class SystemLoginClient:
    def __init__(self):
        self.login_url = getattr(
            settings,
            "chatbot_system_login_url"
        )
        self.api_key = getattr(settings, "chatbot_api_key", "123")

    async def login(self, username: str, password: str) -> dict:
        headers = {
            "Content-Type": "application/json",
            "CHatBot-Key": self.api_key
        }

        payload = {
            "UserName": username,
            "Password": password
        }

        try:
            async with httpx.AsyncClient(timeout=30) as client:
                response = await client.post(
                    self.login_url,
                    json=payload,
                    headers=headers
                )

            response.raise_for_status()
            data = response.json()

            if not data.get("success", False):
                return {
                    "success": False,
                    "error": data.get("error", "Invalid username or password")
                }

            user_context_raw = data.get("userContext")

            if isinstance(user_context_raw, str):
                user_context = json.loads(user_context_raw)
            else:
                user_context = user_context_raw or {}

            return {
                "success": True,
                "raw": data,
                "user_context": user_context
            }

        except Exception as e:
            logger.error(f"SystemLogin failed: {e}")
            return {
                "success": False,
                "error": str(e)
            }


system_login_client = SystemLoginClient()

In [7]:
auth_data = await system_login_client.login(username='evision', password='123')

2026-09-23 07:33:48.562 | ERROR    | __main__:login:57 - SystemLogin failed: Client error '404 Not Found' for url 'http://136.243.82.33:8040/api/ChatBot/SystemLogin'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404


In [8]:
auth_data

{'success': False,
 'error': "Client error '404 Not Found' for url 'http://136.243.82.33:8040/api/ChatBot/SystemLogin'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404"}

In [41]:
# src/tools/system_access_context.py

def extract_allowed_node_ids(user_context: dict) -> list[str]:
    value = (
        user_context.get("AllowedNodeIds")
        or user_context.get("allowedNodeIds")
        or []
    )

    if isinstance(value, str):
        return [x.strip() for x in value.split(",") if x.strip()]

    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]

    return []


def extract_allowed_rep_ids(user_context: dict) -> list[str]:
    value = (
        user_context.get("AllowedRepIds")
        or user_context.get("allowedRepIds")
        or []
    )

    if isinstance(value, str):
        return [x.strip() for x in value.split(",") if x.strip()]

    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]

    return []


def extract_system_user_role(user_context: dict) -> str:
    if user_context.get("AllowedRepIds") or user_context.get("allowedRepIds"):
        return "sales_user"

    return "system_user"


def extract_system_display_name(user_context: dict) -> str:
    return (
        user_context.get("UserName")
        or user_context.get("userName")
        or "System User"
    )

In [42]:
extract_allowed_rep_ids(auth_data['user_context'])

['43',
 '45',
 '103',
 '105',
 '47',
 '49',
 '65',
 '99',
 '30',
 '31',
 '51',
 '53',
 '8',
 '9',
 '10',
 '55',
 '73',
 '93',
 '85',
 '95',
 '109',
 '111',
 '115',
 '117',
 '59',
 '61',
 '83',
 '87',
 '89',
 '91',
 '97',
 '32',
 '33',
 '34',
 '26',
 '27',
 '28',
 '29',
 '39',
 '57',
 '63',
 '67',
 '107',
 '113']

In [43]:
extract_allowed_node_ids(auth_data['user_context'])

['1',
 '2',
 '77',
 '14',
 '69',
 '40',
 '43',
 '44',
 '45',
 '102',
 '103',
 '104',
 '105',
 '70',
 '46',
 '47',
 '48',
 '49',
 '64',
 '65',
 '98',
 '99',
 '75',
 '20',
 '30',
 '21',
 '31',
 '50',
 '51',
 '52',
 '53',
 '78',
 '6',
 '68',
 '7',
 '8',
 '11',
 '9',
 '12',
 '10',
 '54',
 '55',
 '72',
 '73',
 '92',
 '93',
 '79',
 '35',
 '76',
 '84',
 '85',
 '94',
 '95',
 '108',
 '109',
 '110',
 '111',
 '114',
 '115',
 '116',
 '117',
 '80',
 '15',
 '71',
 '58',
 '59',
 '60',
 '61',
 '82',
 '83',
 '86',
 '87',
 '88',
 '89',
 '90',
 '91',
 '96',
 '97',
 '100',
 '101',
 '22',
 '32',
 '24',
 '33',
 '25',
 '34',
 '81',
 '13',
 '74',
 '16',
 '26',
 '17',
 '27',
 '18',
 '28',
 '19',
 '29',
 '38',
 '39',
 '56',
 '57',
 '62',
 '63',
 '66',
 '67',
 '106',
 '107',
 '112',
 '113']

In [44]:
extract_system_user_role(auth_data['user_context'])

'sales_user'